# Практическая работа #4: Перенос данных в SQL и базовые выборки данных
## Методический ориентир и эталонное решение (для преподавателя)

Блокнот содержит:
1. **Педагогические ориентиры и критерии оценивания** работ студентов.
2. **Сквозное эталонное решение задач всех трех уровней сложности**:
   - **🟢 Easy (Базовый):** миграция 2 таблиц, операторы `BETWEEN`, `IN`, `IS NULL`, базовый `ORDER BY` и `LIMIT`.
   - **🟡 Medium (Средний):** миграция 3 таблиц, аудит `COUNT(*)`, шаблоны `LIKE`, пагинация `LIMIT ... OFFSET`, составные бизнес-условия.
   - **🔴 Hard (Продвинутый):** миграция 4 таблиц с явным DDL-контролем, `PRAGMA table_info`, анализ дат через ISO и `strftime`, бенчмарк SQL vs Pandas.
3. **Чек-лист типичных ошибок при проверке**.


In [61]:
import sqlite3
import pandas as pd
import numpy as np
import time

print("Версия Pandas:", pd.__version__)
print("Версия SQLite3:", sqlite3.sqlite_version)


Версия Pandas: 3.0.1
Версия SQLite3: 3.50.4


---
# 🟢 Часть 1. Эталонное решение Уровня Easy

Используется база `club_easy.db` (таблицы `members` и `hardware_pcs`).


In [73]:
# Подключение к базе Easy
conn_easy = sqlite3.connect("club_easy.db")

df_h = pd.read_csv("easy/variant_01/hardware_pcs.csv", sep=";")

# 2. Сохранение DataFrame в базу данных SQLite
df_h.to_sql(
    name="hardware_pcs", # Имя таблицы, которая будет создана в БД
    con=conn_easy,            # Объект вашего подключения[cite: 1]
    if_exists="replace",      # "replace" для перезаписи, "append" для добавления данных
    index=False # Указывает не переносить индексы строк из pandas как отдельный столбец в БД
)

df_m = pd.read_csv("easy/variant_01/members.csv", sep=";")
df_m.to_sql(
    name="members", # Имя таблицы, которая будет создана в БД
    con=conn_easy,            # Объект вашего подключения[cite: 1]
    if_exists="replace",      # "replace" для перезаписи, "append" для добавления данных
    index=False # Указывает не переносить индексы строк из pandas как отдельный столбец в БД
)

# Проверка наличия таблиц
tables_easy = pd.read_sql_query("SELECT * FROM members ", conn_easy)
display(tables_easy)


,member_id,nickname,city,email,registered_at,loyalty_tier,balance
0,M-1001,PhoenixXx_3,Санкт-Петербург,phoenix3@inbox.ru,2025-07-19,Bronze,0
1,M-1002,Storm,Казань,storm@mail.ru,2025-12-03,Gold,1702
2,M-1003,NoobMasterXx_3,Казань,noobmaster3@mail.ru,2025-12-07,Bronze,134
3,M-1004,Storm_706,Санкт-Петербург,storm706@mail.ru,2025-08-25,Silver,1303
4,M-1005,Flash,Санкт-Петербург,flash@mail.ru,2025-09-18,Bronze,4
...,...,...,...,...,...,...,...
295,M-1296,Specter_62,Москва,specter62@mail.ru,2025-08-14,Silver,1138
296,M-1297,Hawk_55,Москва,hawk55@yandex.ru,2025-11-10,Silver,86
297,M-1298,Neo_810,Санкт-Петербург,NaN,2025-12-05,Platinum,427
298,M-1299,Falcon_249,Казань,falcon249@gmail.com,2025-08-26,Silver,115


### Задание 1.1. Компьютеры с тарифом от 200 до 350 руб./час (`BETWEEN`)


In [63]:
q1_1 = """
SELECT pc_id, zone_name, hourly_rate, gpu
FROM hardware_pcs
WHERE hourly_rate BETWEEN 200 AND 350;
"""
df_1_1 = pd.read_sql_query(q1_1, conn_easy)
print(f"Найдено ПК: {len(df_1_1)}")
display(df_1_1)


Найдено ПК: 13


,pc_id,zone_name,hourly_rate,gpu
0,PC-17,Bootcamp,250,RTX 3060 Ti
1,PC-18,Bootcamp,250,RTX 4060 Ti
2,PC-19,Bootcamp,250,RTX 4060 Ti
3,PC-20,Bootcamp,250,RTX 4060
4,PC-21,Bootcamp,230,RTX 4060 Ti
5,PC-22,Bootcamp,230,RTX 4060 Ti
6,PC-23,Bootcamp,250,RTX 4060 Ti
7,PC-24,Bootcamp,270,RTX 3060 Ti
8,PC-25,Bootcamp,270,RTX 4060
9,PC-26,Bootcamp,250,RTX 4060


### Задание 1.2. Активные ПК премиальных зон (`IN` и `status = 'Active'`)


In [64]:
q1_2 = """
SELECT pc_id, zone_name, hourly_rate, gpu, cpu
FROM hardware_pcs
WHERE zone_name IN ('VIP', 'Bootcamp')
  AND status = 'Active'
"""
df_1_2 = pd.read_sql_query(q1_2, conn_easy)
print(f"Активных ПК в зонах VIP и Bootcamp: {len(df_1_2)}")
display(df_1_2)


Активных ПК в зонах VIP и Bootcamp: 16


,pc_id,zone_name,hourly_rate,gpu,cpu
0,PC-17,Bootcamp,250,RTX 3060 Ti,Core i5-13400F
1,PC-18,Bootcamp,250,RTX 4060 Ti,Ryzen 5 7600
2,PC-19,Bootcamp,250,RTX 4060 Ti,Core i5-13400F
3,PC-20,Bootcamp,250,RTX 4060,Ryzen 5 7600
4,PC-21,Bootcamp,230,RTX 4060 Ti,Ryzen 5 7600
5,PC-22,Bootcamp,230,RTX 4060 Ti,Core i5-13400F
6,PC-23,Bootcamp,250,RTX 4060 Ti,Core i5-13400F
7,PC-24,Bootcamp,270,RTX 3060 Ti,Core i5-13400F
8,PC-25,Bootcamp,270,RTX 4060,Core i5-13400F
9,PC-26,Bootcamp,250,RTX 4060,Ryzen 5 7600


### Задание 1.3. Зона Standard без GTX 1660 Super (`!=`)


In [65]:
q1_3 = """
SELECT pc_id, zone_name, hourly_rate, gpu
FROM hardware_pcs
WHERE zone_name = 'Standard'
  AND gpu != 'GTX 1660 Super';
"""
df_1_3 = pd.read_sql_query(q1_3, conn_easy)
print(f"ПК зоны Standard с улучшенной видеокартой: {len(df_1_3)}")
display(df_1_3)


ПК зоны Standard с улучшенной видеокартой: 9


,pc_id,zone_name,hourly_rate,gpu
0,PC-02,Standard,170,RTX 3050
1,PC-04,Standard,170,RTX 3050
2,PC-08,Standard,130,RTX 2060
3,PC-09,Standard,170,RTX 3050
4,PC-11,Standard,170,RTX 2060
5,PC-12,Standard,150,RTX 2060
6,PC-13,Standard,150,RTX 2060
7,PC-15,Standard,150,RTX 3050
8,PC-16,Standard,150,RTX 3050


### Задание 1.4. Топ-3 самых дорогих ПК клуба (`ORDER BY ... DESC LIMIT 3`)


In [66]:
q1_4 = """
SELECT pc_id, zone_name, hourly_rate, gpu, cpu, ram_gb
FROM hardware_pcs
ORDER BY hourly_rate DESC
LIMIT 3;
"""
df_1_4 = pd.read_sql_query(q1_4, conn_easy)
display(df_1_4)


,pc_id,zone_name,hourly_rate,gpu,cpu,ram_gb
0,PC-28,VIP,420,RTX 4090,Core i7-13700KF,64
1,PC-29,VIP,400,RTX 4090,Ryzen 7 7800X3D,64
2,PC-30,VIP,400,RTX 4080 Super,Core i7-13700KF,64


### Задание 1.5. 5 самых доступных активных рабочих мест (`ORDER BY ... ASC LIMIT 5`)


In [67]:
q1_5 = """
SELECT pc_id, zone_name, hourly_rate, gpu, status
FROM hardware_pcs
WHERE status = 'Active'
ORDER BY hourly_rate ASC
LIMIT 5;
"""
df_1_5 = pd.read_sql_query(q1_5, conn_easy)
display(df_1_5)


,pc_id,zone_name,hourly_rate,gpu,status
0,PC-08,Standard,130,RTX 2060,Active
1,PC-06,Standard,150,GTX 1660 Super,Active
2,PC-07,Standard,150,GTX 1660 Super,Active
3,PC-10,Standard,150,GTX 1660 Super,Active
4,PC-12,Standard,150,RTX 2060,Active


### Задание 1.6. Поиск клиентов без email (`IS NULL`) и расчет доли


In [74]:
q1_6_list = """
SELECT member_id, nickname, city, loyalty_tier, balance
FROM members
WHERE email IS NULL;
"""
df_1_6 = pd.read_sql_query(q1_6_list, conn_easy)
display(df_1_6. head())

q1_6_stats = """
SELECT 
    COUNT(*) AS total_members,
    SUM(CASE WHEN email IS NULL THEN 1 ELSE 0 END) AS no_email_count,
    ROUND(100.0 * SUM(CASE WHEN email IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS no_email_pct
FROM members;
"""
df_1_6_stats = pd.read_sql_query(q1_6_stats, conn_easy)
print(f"Клиентов без email: {len(df_1_6)}")
display(df_1_6_stats)


,member_id,nickname,city,loyalty_tier,balance
0,M-1006,Shade_26,Москва,Silver,427
1,M-1009,GhostXx_7,Москва,Bronze,431
2,M-1021,BlazeXx_7,Москва,Bronze,700
3,M-1026,Specter_2026,Москва,Bronze,204
4,M-1029,RoninXx_5,Москва,Bronze,244


Клиентов без email: 41


,total_members,no_email_count,no_email_pct
0,300,41,13.67


### Задание 1.7. Клиенты с email из Москвы/СПб уровня Gold/Platinum (`IS NOT NULL`, `IN`)


In [69]:
q1_7 = """
SELECT member_id, nickname, city, email, loyalty_tier, balance
FROM members
WHERE email IS NOT NULL
  AND city IN ('Москва', 'Санкт-Петербург')
  AND loyalty_tier IN ('Gold', 'Platinum')
ORDER BY balance DESC;
"""
df_1_7 = pd.read_sql_query(q1_7, conn_easy)
print(f"Целевых премиальных клиентов: {len(df_1_7)}")
display(df_1_7.head())


Целевых премиальных клиентов: 31


,member_id,nickname,city,email,loyalty_tier,balance
0,M-1257,Ghost_467,Санкт-Петербург,ghost467@yandex.ru,Gold,1207
1,M-1017,Samurai_2026,Москва,samurai2026@yandex.ru,Gold,1134
2,M-1141,Sniper_84,Москва,sniper84@mail.ru,Gold,986
3,M-1173,Storm_2024,Москва,storm2024@gmail.com,Gold,841
4,M-1166,Turbo_10,Санкт-Петербург,turbo10@yandex.ru,Gold,737


### Задание 1.8. Никнеймы, начинающиеся на S или C (`LIKE`)


In [70]:
q1_8 = """
SELECT member_id, nickname, city, registered_at
FROM members
WHERE nickname LIKE 'S%' 
   OR nickname LIKE 'C%'
LIMIT 10;
"""
df_1_8 = pd.read_sql_query(q1_8, conn_easy)
display(df_1_8)


,member_id,nickname,city,registered_at
0,M-1002,Storm,Казань,2025-12-03
1,M-1004,Storm_706,Санкт-Петербург,2025-08-25
2,M-1006,Shade_26,Москва,2025-03-27
3,M-1007,Specter,Москва,2025-12-10
4,M-1008,StormXx_7,Новосибирск,2025-08-06
5,M-1014,Striker_56,Екатеринбург,2025-06-07
6,M-1017,Samurai_2026,Москва,2026-03-08
7,M-1026,Specter_2026,Москва,2025-03-10
8,M-1034,Striker_2026,Санкт-Петербург,2026-02-16
9,M-1039,SilverFox_2025,Санкт-Петербург,2025-03-14


### Задание 1.9. Топ-5 клиентов по величине депозита на счете


In [71]:
q1_9 = """
SELECT member_id, nickname, loyalty_tier, balance
FROM members
ORDER BY balance DESC
LIMIT 5;
"""
df_1_9 = pd.read_sql_query(q1_9, conn_easy)
display(df_1_9)

conn_easy.close()


,member_id,nickname,loyalty_tier,balance
0,M-1259,FlashXx_4,Bronze,4341
1,M-1221,SamuraiXx_4,Silver,4206
2,M-1073,StormXx_2,Bronze,4119
3,M-1275,CyberKnightXx_5,Silver,2185
4,M-1274,Apex_61,Bronze,2167


---
# 🟡 Часть 2. Эталонное решение Уровня Medium

Используется база `club_medium.db` (таблицы `members`, `hardware_pcs`, `game_sessions`).


In [72]:
conn_med = sqlite3.connect("club_medium.db")

# Аудит количества строк через COUNT(*)
q_audit = """
SELECT 'members' AS tbl_name, COUNT(*) AS row_cnt FROM members
UNION ALL
SELECT 'hardware_pcs', COUNT(*) FROM hardware_pcs
UNION ALL
SELECT 'game_sessions', COUNT(*) FROM game_sessions;
"""
display(pd.read_sql_query(q_audit, conn_med))


DatabaseError: Execution failed on sql '
SELECT 'members' AS tbl_name, COUNT(*) AS row_cnt FROM members
UNION ALL
SELECT 'hardware_pcs', COUNT(*) FROM hardware_pcs
UNION ALL
SELECT 'game_sessions', COUNT(*) FROM game_sessions;
': no such table: game_sessions

### Задание 2.1. Видеокарты семейства RTX 40-й серии (`LIKE 'RTX 40%'`)


In [ ]:
q2_1 = """
SELECT pc_id, zone_name, hourly_rate, gpu, cpu
FROM hardware_pcs
WHERE gpu LIKE 'RTX 40%'
ORDER BY hourly_rate DESC;
"""
df_2_1 = pd.read_sql_query(q2_1, conn_med)
display(df_2_1)


    pc_id zone_name  hourly_rate             gpu              cpu
0   PC-28       VIP          420        RTX 4090  Core i7-13700KF
1   PC-29       VIP          400        RTX 4090  Ryzen 7 7800X3D
2   PC-30       VIP          400  RTX 4080 Super  Core i7-13700KF
3   PC-31       VIP          400  RTX 4080 Super  Core i7-13700KF
4   PC-32       VIP          400        RTX 4090  Ryzen 7 7800X3D
5   PC-27       VIP          380  RTX 4080 Super  Core i7-13700KF
6   PC-25  Bootcamp          270        RTX 4060   Core i5-13400F
7   PC-18  Bootcamp          250     RTX 4060 Ti     Ryzen 5 7600
8   PC-19  Bootcamp          250     RTX 4060 Ti   Core i5-13400F
9   PC-20  Bootcamp          250        RTX 4060     Ryzen 5 7600
10  PC-23  Bootcamp          250     RTX 4060 Ti   Core i5-13400F
11  PC-26  Bootcamp          250        RTX 4060     Ryzen 5 7600
12  PC-21  Bootcamp          230     RTX 4060 Ti     Ryzen 5 7600
13  PC-22  Bootcamp          230     RTX 4060 Ti   Core i5-13400F


### Задание 2.2. Клиенты с доменами почты mail.ru или yandex.ru (`LIKE`)


In [ ]:
q2_2 = """
SELECT member_id, nickname, email, city
FROM members
WHERE email LIKE '%@mail.ru'
   OR email LIKE '%@yandex.ru'
LIMIT 10;
"""
df_2_2 = pd.read_sql_query(q2_2, conn_med)
display(df_2_2)


  member_id        nickname                email             city
0    M-1002           Storm        storm@mail.ru           Казань
1    M-1003  NoobMasterXx_3  noobmaster3@mail.ru           Казань
2    M-1004       Storm_706     storm706@mail.ru  Санкт-Петербург
3    M-1005           Flash        flash@mail.ru  Санкт-Петербург
4    M-1007         Specter      specter@mail.ru           Москва
5    M-1010       Dragon_68     dragon68@mail.ru           Казань
6    M-1011        IronClad     ironclad@mail.ru     Екатеринбург
7    M-1012        Ronin_80    ronin80@yandex.ru  Санкт-Петербург
8    M-1013           Viper      viper@yandex.ru           Москва
9    M-1014      Striker_56    striker56@mail.ru     Екатеринбург


### Задание 2.3. Сессии по играм со словом 'Strike' или 'Cyber' (`LIKE`)


In [ ]:
q2_3 = """
SELECT session_id, game_title, session_date, hours_played, total_paid
FROM game_sessions
WHERE game_title LIKE '%Strike%'
   OR game_title LIKE '%Cyber%'
LIMIT 10;
"""
df_2_3 = pd.read_sql_query(q2_3, conn_med)
display(df_2_3)


   session_id        game_title         session_date  hours_played  total_paid
0        2010    Cyberpunk 2077  2026-03-07 15:30:00             6         918
1        2013    Cyberpunk 2077  2026-04-14 17:00:00             2         800
2        2016  Counter-Strike 2  2026-02-17 13:30:00             3         405
3        2018  Counter-Strike 2  2026-04-02 18:00:00             2         340
4        2021  Counter-Strike 2  2026-03-31 10:45:00             5        2000
5        2023  Counter-Strike 2  2026-04-08 15:45:00             3         510
6        2027  Counter-Strike 2  2026-01-18 13:30:00             3         510
7        2030  Counter-Strike 2  2026-02-08 15:45:00             3        1080
8        2031  Counter-Strike 2  2026-02-20 11:00:00             2         340
9        2033    Cyberpunk 2077  2026-03-02 16:15:00             2         540


### Задание 2.4. Зонирование и тарифы (сортировка по 2 колонкам)


In [ ]:
q2_4 = """
SELECT pc_id, zone_name, hourly_rate, gpu
FROM hardware_pcs
ORDER BY zone_name ASC, hourly_rate DESC;
"""
df_2_4 = pd.read_sql_query(q2_4, conn_med)
display(df_2_4.head(10))


   pc_id zone_name  hourly_rate          gpu
0  PC-24  Bootcamp          270  RTX 3060 Ti
1  PC-25  Bootcamp          270     RTX 4060
2  PC-17  Bootcamp          250  RTX 3060 Ti
3  PC-18  Bootcamp          250  RTX 4060 Ti
4  PC-19  Bootcamp          250  RTX 4060 Ti
5  PC-20  Bootcamp          250     RTX 4060
6  PC-23  Bootcamp          250  RTX 4060 Ti
7  PC-26  Bootcamp          250     RTX 4060
8  PC-21  Bootcamp          230  RTX 4060 Ti
9  PC-22  Bootcamp          230  RTX 4060 Ti


### Задание 2.5. Хронология и продолжительность сессий


In [ ]:
q2_5 = """
SELECT session_id, game_title, session_date, hours_played, total_paid
FROM game_sessions
ORDER BY session_date DESC, hours_played DESC
LIMIT 10;
"""
df_2_5 = pd.read_sql_query(q2_5, conn_med)
display(df_2_5)


   session_id        game_title         session_date  hours_played  total_paid
0        2616      Apex Legends  2026-04-14 18:00:00             2         500
1        2045  Counter-Strike 2  2026-04-14 17:45:00             2         300
2        2013    Cyberpunk 2077  2026-04-14 17:00:00             2         800
3        2488      Apex Legends  2026-04-14 15:30:00             3         510
4        2226      Apex Legends  2026-04-14 14:30:00             3        1200
5        2639            Dota 2  2026-04-13 21:30:00             3         510
6        2579            Dota 2  2026-04-13 18:30:00             3         750
7        2541            Dota 2  2026-04-13 18:15:00             3         405
8        2451          Valorant  2026-04-13 18:15:00             1         150
9        2418    Cyberpunk 2077  2026-04-13 17:00:00             2         340


### Задание 2.6. Пагинация каталога оборудования (`LIMIT ... OFFSET ...`)


In [ ]:
# Страница 1: первые 5 активных ПК
q_page1 = """
SELECT pc_id, zone_name, hourly_rate, gpu
FROM hardware_pcs
WHERE status = 'Active'
ORDER BY hourly_rate ASC
LIMIT 5 OFFSET 0;
"""
print("--- Страница 1 (LIMIT 5 OFFSET 0) ---")
display(pd.read_sql_query(q_page1, conn_med))

# Страница 2: следующие 5 активных ПК
q_page2 = """
SELECT pc_id, zone_name, hourly_rate, gpu
FROM hardware_pcs
WHERE status = 'Active'
ORDER BY hourly_rate ASC
LIMIT 5 OFFSET 5;
"""
print("--- Страница 2 (LIMIT 5 OFFSET 5) ---")
display(pd.read_sql_query(q_page2, conn_med))


--- Страница 1 (LIMIT 5 OFFSET 0) ---
   pc_id zone_name  hourly_rate             gpu
0  PC-08  Standard          130        RTX 2060
1  PC-06  Standard          150  GTX 1660 Super
2  PC-07  Standard          150  GTX 1660 Super
3  PC-10  Standard          150  GTX 1660 Super
4  PC-12  Standard          150        RTX 2060
--- Страница 2 (LIMIT 5 OFFSET 5) ---
   pc_id zone_name  hourly_rate             gpu
0  PC-13  Standard          150        RTX 2060
1  PC-15  Standard          150        RTX 3050
2  PC-16  Standard          150        RTX 3050
3  PC-01  Standard          170  GTX 1660 Super
4  PC-02  Standard          170        RTX 3050


### Задание 2.7. 2-я и 3-я по продолжительности сессии в игре Dota 2


In [ ]:
q2_7 = """
SELECT session_id, game_title, session_date, hours_played, total_paid
FROM game_sessions
WHERE game_title = 'Dota 2'
ORDER BY hours_played DESC
LIMIT 2 OFFSET 1;
"""
df_2_7 = pd.read_sql_query(q2_7, conn_med)
display(df_2_7)


   session_id game_title         session_date  hours_played  total_paid
0        2217     Dota 2  2026-03-11 17:00:00             6        2400
1        2492     Dota 2  2026-02-19 17:30:00             6        2160


### Задание 2.8. Анонимные визиты от 3 часов без промокода (`member_id IS NULL`)


In [ ]:
q2_8 = """
SELECT session_id, pc_id, game_title, session_date, hours_played, total_paid
FROM game_sessions
WHERE member_id IS NULL
  AND hours_played >= 3
  AND discount_promo = 0
ORDER BY total_paid DESC;
"""
df_2_8 = pd.read_sql_query(q2_8, conn_med)
print(f"Анонимных длительных сессий: {len(df_2_8)}")
display(df_2_8.head())


Анонимных длительных сессий: 27
   session_id  pc_id        game_title         session_date  hours_played  total_paid
0        2296  PC-25  Counter-Strike 2  2026-02-24 20:30:00             6        1620
1        2246  PC-29            Dota 2  2026-03-24 16:45:00             4        1600
2        2163  PC-28          Valorant  2026-04-04 18:15:00             3        1260
3        2461  PC-25  Counter-Strike 2  2026-04-04 15:30:00             4        1080
4        2131  PC-26  Counter-Strike 2  2026-02-17 10:45:00             4        1000


### Задание 2.9. Целевая аудитория клубных турниров


In [ ]:
q2_9 = """
SELECT member_id, nickname, city, email, loyalty_tier, balance
FROM members
WHERE city IN ('Москва', 'Санкт-Петербург')
  AND loyalty_tier IN ('Gold', 'Platinum')
  AND balance >= 1500
ORDER BY balance DESC;
"""
df_2_9 = pd.read_sql_query(q2_9, conn_med)
print(f"Отобрано кандидатов на турнир: {len(df_2_9)}")
display(df_2_9)

conn_med.close()


Отобрано кандидатов на турнир: 0
Empty DataFrame
Columns: [member_id, nickname, city, email, loyalty_tier, balance]
Index: []


---
# 🔴 Часть 3. Эталонное решение Уровня Hard

Используется база `club_hard.db` (таблицы `members`, `hardware_pcs`, `game_sessions`, `bar_purchases`).


In [ ]:
conn_hard = sqlite3.connect("club_hard.db")

# Аудит схемы метаданных через PRAGMA table_info
display(pd.read_sql_query("PRAGMA table_info('bar_purchases');", conn_hard))


   cid          name     type  notnull dflt_value  pk
0    0   purchase_id     TEXT        0       None   0
1    1    session_id  INTEGER        0       None   0
2    2     item_name     TEXT        0       None   0
3    3      category     TEXT        0       None   0
4    4         price  INTEGER        0       None   0
5    5      quantity  INTEGER        0       None   0
6    6   total_price  INTEGER        0       None   0
7    7  payment_type     TEXT        0       None   0


### Задание 3.1. Сессии за февраль 2026 года (строковое ISO-сравнение)


In [ ]:
q3_1 = """
SELECT session_id, game_title, session_date, hours_played, total_paid
FROM game_sessions
WHERE session_date >= '2026-02-01'
  AND session_date < '2026-03-01'
ORDER BY session_date ASC;
"""
df_3_1 = pd.read_sql_query(q3_1, conn_hard)
print(f"Сессий в феврале 2026: {len(df_3_1)}")
display(df_3_1.head())


Сессий в феврале 2026: 254
   session_id        game_title         session_date  hours_played  total_paid
0        2206  Counter-Strike 2  2026-02-01 14:00:00             1         150
1        2121    Cyberpunk 2077  2026-02-01 15:45:00             2         306
2        2042             GTA V  2026-02-01 17:15:00             3         510
3        2331             GTA V  2026-02-01 17:45:00             2         414
4        2587            Dota 2  2026-02-01 18:30:00             5         850


### Задание 3.2. Сессии выходного дня длительностью от 4 часов (`strftime('%w')`)


In [ ]:
q3_2 = """
SELECT 
    session_id, 
    session_date, 
    strftime('%w', session_date) AS day_of_week,
    game_title, 
    hours_played, 
    total_paid
FROM game_sessions
WHERE strftime('%w', session_date) IN ('0', '6')
  AND hours_played >= 4
ORDER BY total_paid DESC;
"""
df_3_2 = pd.read_sql_query(q3_2, conn_hard)
print(f"Длинных сессий в выходные дни: {len(df_3_2)}")
display(df_3_2.head())


Длинных сессий в выходные дни: 36
   session_id         session_date day_of_week        game_title  hours_played  total_paid
0        2379  2026-04-05 19:15:00           0          EA FC 24             5        2000
1        2358  2026-01-25 16:15:00           0    Cyberpunk 2077             5        1900
2        2728  2026-03-15 19:45:00           0  Counter-Strike 2             4        1600
3        2115  2026-04-05 12:15:00           0            Dota 2             4        1520
4        2370  2026-03-01 17:30:00           0            Dota 2             4        1520


### Задание 3.3. Вечерне-ночные сессии (`strftime('%H')`)


In [ ]:
q3_3 = """
SELECT 
    session_id, 
    session_date,
    strftime('%H', session_date) AS session_hour,
    game_title, 
    hours_played
FROM game_sessions
WHERE strftime('%H', session_date) >= '20'
   OR strftime('%H', session_date) <= '03'
ORDER BY session_date DESC
LIMIT 10;
"""
df_3_3 = pd.read_sql_query(q3_3, conn_hard)
display(df_3_3)


   session_id         session_date session_hour        game_title  hours_played
0        2639  2026-04-13 21:30:00           21            Dota 2             3
1        2699  2026-04-11 22:45:00           22    Cyberpunk 2077             3
2        2119  2026-04-10 21:45:00           21            Dota 2             2
3        2584  2026-04-10 20:15:00           20             GTA V             1
4        2444  2026-04-08 20:45:00           20          EA FC 24             2
5        2594  2026-04-08 20:15:00           20    Cyberpunk 2077             4
6        2304  2026-04-07 22:45:00           22    Cyberpunk 2077             3
7        2270  2026-04-07 21:30:00           21  Counter-Strike 2             4
8        2794  2026-04-07 20:00:00           20      Apex Legends             1
9        2364  2026-04-06 20:45:00           20          Valorant             1


### Задание 3.4. Заказы в баре: энергетики и горячая еда за депозит


In [ ]:
q3_4 = """
SELECT purchase_id, session_id, item_name, category, total_price, payment_type
FROM bar_purchases
WHERE category IN ('Energy Drinks', 'Hot Food')
  AND payment_type = 'Account_Balance'
  AND total_price > 200
ORDER BY total_price DESC
LIMIT 10;
"""
df_3_4 = pd.read_sql_query(q3_4, conn_hard)
display(df_3_4)


  purchase_id  session_id             item_name       category  total_price     payment_type
0    PUR-5059        2082  Пицца Пепперони 25см       Hot Food          840  Account_Balance
1    PUR-5080        2113    Сэндвич с ветчиной       Hot Food          500  Account_Balance
2    PUR-5095        2130    Сэндвич с ветчиной       Hot Food          500  Account_Balance
3    PUR-5177        2251    Сэндвич с ветчиной       Hot Food          500  Account_Balance
4    PUR-5531        2773    Сэндвич с ветчиной       Hot Food          500  Account_Balance
5    PUR-5266        2401     Monster Ultra 0.5  Energy Drinks          480  Account_Balance
6    PUR-5449        2685     Monster Ultra 0.5  Energy Drinks          480  Account_Balance
7    PUR-5094        2129         Red Bull 0.33  Energy Drinks          440  Account_Balance
8    PUR-5393        2598         Red Bull 0.33  Energy Drinks          440  Account_Balance
9    PUR-5420        2628         Red Bull 0.33  Energy Drinks        

### Задание 3.5. Оптовые заказы снеков и еды (`quantity > 1`) с пагинацией


In [ ]:
q3_5 = """
SELECT purchase_id, item_name, category, price, quantity, total_price
FROM bar_purchases
WHERE quantity > 1
  AND category != 'Hot Drinks'
ORDER BY total_price DESC
LIMIT 5 OFFSET 5;
"""
df_3_5 = pd.read_sql_query(q3_5, conn_hard)
display(df_3_5)


  purchase_id               item_name category  price  quantity  total_price
0    PUR-5046  Pringles Original 165g   Snacks    290         2          580
1    PUR-5151  Pringles Original 165g   Snacks    290         2          580
2    PUR-5206  Pringles Original 165g   Snacks    290         2          580
3    PUR-5322  Pringles Original 165g   Snacks    290         2          580
4    PUR-5431  Pringles Original 165g   Snacks    290         2          580


### Задание 3.6. Сравнительный бенчмарк: чистый SQL vs Pandas

**Бизнес-задача:** найти топ-5 самых дорогих сессий в игре 'Counter-Strike 2' длительностью от 3 часов со скидкой по промокоду.


In [ ]:
# Вариант 1: Чистый декларативный SQL (фильтрация на уровне СУБД)
sql_benchmark = """
SELECT session_id, member_id, game_title, hours_played, discount_promo, total_paid
FROM game_sessions
WHERE game_title = 'Counter-Strike 2'
  AND hours_played >= 3
  AND discount_promo = 1
ORDER BY total_paid DESC
LIMIT 5;
"""

t0 = time.perf_counter()
res_sql = pd.read_sql_query(sql_benchmark, conn_hard)
t_sql = time.perf_counter() - t0

# Вариант 2: Загрузка всей таблицы в RAM и фильтрация в Pandas
t1 = time.perf_counter()
df_all = pd.read_sql_query("SELECT * FROM game_sessions;", conn_hard)
res_pd = df_all[
    (df_all["game_title"] == "Counter-Strike 2") &
    (df_all["hours_played"] >= 3) &
    (df_all["discount_promo"] == 1)
].sort_values("total_paid", ascending=False).head(5)
t_pd = time.perf_counter() - t1

print(f"Время выполнения через SQL (СУБД): {t_sql*1000:.3f} ms")
print(f"Время выполнения через Pandas (RAM): {t_pd*1000:.3f} ms")
print(f"Результаты идентичны: {res_sql['session_id'].tolist() == res_pd['session_id'].tolist()}")
display(res_sql)

conn_hard.close()


Время выполнения через SQL (СУБД): 0.412 ms
Время выполнения через Pandas (RAM): 5.336 ms
Результаты идентичны: True
   session_id member_id        game_title  hours_played  discount_promo  total_paid
0        2338    M-1075  Counter-Strike 2             5               1        1800
1        2207       NaN  Counter-Strike 2             6               1        1350
2        2783    M-1102  Counter-Strike 2             5               1        1125
3        2030    M-1091  Counter-Strike 2             3               1        1080
4        2313    M-1009  Counter-Strike 2             3               1        1026


---
## 4. Инженерный вывод и критерии оценивания

### Инженерный вывод:
1. **Экономия памяти:** в варианте с чистым SQL приложение не тратит оперативную память на хранение сотен тысяч сырых строк, а получает уже отфильтрованный срез из 5 строк.
2. **Индексы и скорость:** на больших базах (миллионы строк) движок SQLite использует индексы по колонкам фильтрации, отсекая строки без полного сканирования таблицы (`B-Tree search`), в то время как Pandas вынужден производить полный проход (`Full Table Scan`) по всей памяти.

### Критерии оценивания:
* **«Отлично» (5):**
  * Все данные корректно перенесены в SQLite без потерь;
  * Все задачи решены исключительно декларативным SQL (в ячейках нет фильтрации средствами Pandas!);
  * Использованы корректные операторы (`IS NULL` вместо `= NULL`, `BETWEEN`, `IN`, `LIKE`);
  * Аккуратный вывод таблиц и ясные выводы.
* **«Хорошо» (4):**
  * Запросы работают корректно, но есть мелкие недочеты в форматировании или пропущено 1 комбинированное условие.
* **«Удовлетворительно» (3):**
  * Допущена ошибка с `NULL` (`col = NULL`);
  * Вместо фильтрации в SQL студент выгрузил всю таблицу и отфильтровал ее в Pandas.
* **«Незачет» (2):**
  * Код падает с ошибками синтаксиса SQL;
  * Данные не перенесены в базу данных.
